# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv("data/AviationData.csv", encoding="latin-1")
df["Event.Date"] = pd.to_datetime(df["Event.Date"])

print(df.shape)
print("Date range:", df["Event.Date"].min(), "to", df["Event.Date"].max())
df.head()

(88889, 31)
Date range: 1948-10-24 00:00:00 to 2022-12-29 00:00:00


C:\Users\jeanf\AppData\Local\Temp\ipykernel_33580\2709654219.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="latin-1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
print(df.dtypes)
print(df.isna().sum())

Event.Id                          object
Investigation.Type                object
Accident.Number                   object
Event.Date                datetime64[ns]
Location                          object
Country                           object
Latitude                          object
Longitude                         object
Airport.Code                      object
Airport.Name                      object
Injury.Severity                   object
Aircraft.damage                   object
Aircraft.Category                 object
Registration.Number               object
Make                              object
Model                             object
Amateur.Built                     object
Number.of.Engines                float64
Engine.Type                       object
FAR.Description                   object
Schedule                          object
Purpose.of.flight                 object
Air.carrier                       object
Total.Fatal.Injuries             float64
Total.Serious.In

In [4]:
df.describe()

,Event.Date,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,88889,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1999-09-17 17:13:39.354475904,1.146585,0.647855,0.279881,0.357061,5.325440
min,1948-10-24 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1989-01-15 00:00:00,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1998-07-18 00:00:00,1.000000,0.000000,0.000000,0.000000,1.000000
75%,2009-07-01 00:00:00,1.000000,0.000000,0.000000,0.000000,2.000000
max,2022-12-29 00:00:00,8.000000,349.000000,161.000000,380.000000,699.000000
std,NaN,0.446510,5.485960,1.544084,2.235625,27.913634


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [5]:
# 1
df = df[df["Event.Date"] >= "1983-01-01"]

# 2
df = df[df["Amateur.Built"] == "No"]

# 3
df = df.dropna(subset=["Make", "Model"])

# 4
def clean_text(text):
    return text.strip().upper()

df["Make"] = df["Make"].apply(clean_text)
df["Model"] = df["Model"].apply(clean_text)

print(df.shape)
print(df["Aircraft.Category"].value_counts())

(76888, 31)
Aircraft.Category
Airplane             21428
Helicopter            3021
Glider                 434
Balloon                199
Weight-Shift           139
Powered Parachute       83
Gyrocraft               31
Ultralight              18
Unknown                 11
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
Name: count, dtype: int64


In [6]:
# Standardizing the Model labels
def clean_model(model):
    return model.replace(" ", "").replace("-", "").replace("/", "")

df["Model"] = df["Model"].apply(clean_model)

# Creation of a unique plane type: Make + Model
df["Make.Model"] = df["Make"] + " " + df["Model"]

# Building a lookup from the rows where the category is known
known = df.dropna(subset=["Aircraft.Category"])
known_models = list(known["Make.Model"])
known_categories = list(known["Aircraft.Category"])
category_lookup = {}
for i in range(len(known_models)):
    category_lookup[known_models[i]] = known_categories[i]

# Filling the missing categories using the lookup
df["Aircraft.Category"] = df["Aircraft.Category"].fillna(df["Make.Model"].map(category_lookup))
print("Still missing:", df["Aircraft.Category"].isna().sum())


df = df[df["Aircraft.Category"] == "Airplane"]
print(df.shape)

Still missing: 6134
(62095, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [7]:
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
print(df[injury_cols].isna().sum())
df[injury_cols] = df[injury_cols].fillna(0)

# Estimated number of people on board = fatal + serious + minor + uninjured
df["Total.Passengers"] = df[injury_cols].sum(axis=1)

# Events with nobody recorded say nothing about injury rates so we drop them
print("Events with no people recorded:", (df["Total.Passengers"] == 0).sum())
df = df[df["Total.Passengers"] > 0]

# Fraction of the people on board that were fatally or seriously injured
df["Fatal.Serious.Fraction"] = (df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]) / df["Total.Passengers"]
df["Fatal.Serious.Fraction"].describe()

Total.Fatal.Injuries      7988
Total.Serious.Injuries    8794
Total.Minor.Injuries      8312
Total.Uninjured           3685
dtype: int64
Events with no people recorded: 1073


count    61022.000000
mean         0.269248
std          0.428063
min          0.000000
25%          0.000000
50%          0.000000
75%          0.666667
max          1.000000
Name: Fatal.Serious.Fraction, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [8]:
# Rename the column so it matches the name used in the instructions
df = df.rename(columns={"Aircraft.damage": "Aircraft.Damage"})
print(df["Aircraft.Damage"].value_counts())

# "Unknown" means we don't know the damage so we treat it as missing and drop those rows
df["Aircraft.Damage"] = df["Aircraft.Damage"].replace("Unknown", np.nan)
df = df.dropna(subset=["Aircraft.Damage"])

# Derived column: 1 if the aircraft was destroyed, 0 otherwise
df["Destroyed"] = (df["Aircraft.Damage"] == "Destroyed").astype(int)
df["Destroyed"].value_counts()

Aircraft.Damage
Substantial    45617
Destroyed      12151
Minor           1567
Unknown           74
Name: count, dtype: int64


Destroyed
0    47184
1    12151
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [9]:

def clean_make(name):
    return name.replace(".", "").replace(",", "")

df["Make"] = df["Make"].apply(clean_make)

# Spelling variants of the same manufacturer found by inspecting value_counts
variants = {
    "PIPER": ["PIPER AIRCRAFT INC", "PIPER AIRCRAFT CORPORATION", "NEW PIPER AIRCRAFT INC", "PIPER AIRCRAFT", "NEW PIPER"],
    "BEECH": ["BEECHCRAFT", "HAWKER BEECHCRAFT CORP", "HAWKER BEECHCRAFT", "HAWKER BEECHCRAFT CORPORATION", "BEECH AIRCRAFT CORPORATION",
              "HAWKER-BEECHCRAFT", "HAWKER BEECH", "BEECH AIRCRAFT", "BEECH AIRCRAFT CORP", "BEECHCRAFT CORPORATION", "BEECH AIRCRAFT CO"],
    "CESSNA": ["CESSNA AIRCRAFT CO", "CESSNA AIRCRAFT COMPANY", "CESSNA AIRCRAFT", "REIMS-CESSNA"],
    "BOEING": ["BOEING COMPANY", "THE BOEING COMPANY"],
    "BOEING STEARMAN": ["BOEING-STEARMAN"],
    "AIRBUS": ["AIRBUS INDUSTRIE", "AIRBUS INDUSTRIES"],
    "MCDONNELL DOUGLAS": ["MCDONNELL DOUGLAS AIRCRAFT CO", "MCDONNELL DOUGLAS CORPORATION", "MCDONNELL-DOUGLAS"],
    "DE HAVILLAND": ["DEHAVILLAND", "DEHAVILLAND CANADA"],
    "GRUMMAN": ["GRUMMAN AMERICAN", "GRUMMAN AMERICAN AVN CORP", "GRUMMAN AMERICAN AVIATION", "GRUMMAN AMERICAN AVIATION CORP",
                "GRUMMAN AMERICAN CORPORATION", "GRUMMAN AIRCRAFT", "GRUMMAN ACFT ENG", "GRUMMAN AIRCRAFT ENG CORP"],
    "SCHWEIZER": ["GRUMMAN-SCHWEIZER", "GRUMMAN ACFT ENG COR-SCHWEIZER", "GULFSTREAM-SCHWEIZER", "SCHWEIZER AIRCRAFT CORP",
                  "GULFSTREAM-SCHWEIZER A/C CORP", "GRUMMAN SCHWEIZER", "GRUMMAN AIRCRAFT COR-SCHWEIZER", "GULFSTREAM SCHWEIZER A/C CORP"],
    "CIRRUS": ["CIRRUS DESIGN CORP", "CIRRUS DESIGN CORPORATION", "CIRRUS DESIGN"],
    "AIR TRACTOR": ["AIR TRACTOR INC"],
    "EMBRAER": ["EMBRAER S A", "EMBRAER-EMPRESA BRASILEIRA DE", "EMBRAER SA"],
    "LEARJET": ["GATES LEARJET", "LEARJET INC", "GATES LEARJET CORP", "GATES LEARJET CORPORATION"],
    "BOMBARDIER": ["BOMBARDIER INC", "BOMBARDIER AEROSPACE INC"],
    "ROCKWELL": ["ROCKWELL INTERNATIONAL", "NORTH AMERICAN ROCKWELL CORP", "NORTH AMERICAN ROCKWELL"],
    "MOONEY": ["MOONEY AIRCRAFT CORP", "MOONEY AIRPLANE CO INC", "MOONEY AIRCRAFT", "MOONEY AIRPLANE COMPANY INC", "MOONEY AIRCRAFT CORPORATION"],
    "SAAB": ["SAAB-SCANIA AB (SAAB)", "SAAB-SCANIA", "SAAB-SCANIA AB"],
    "GULFSTREAM": ["GULFSTREAM AMERICAN", "GULFSTREAM AEROSPACE", "GULFSTREAM AMERICAN CORP", "GULFSTREAM AEROSPACE CORP", "GULFSTREAM AEROSPACE LP"],
    "RAYTHEON": ["RAYTHEON AIRCRAFT COMPANY", "RAYTHEON CORPORATE JETS", "RAYTHEON CO", "RAYTHEON COMPANY"],
    "DIAMOND": ["DIAMOND AIRCRAFT IND INC", "DIAMOND AIRCRAFT INDUSTRIES", "DIAMOND AIRCRAFT IND GMBH", "DIAMOND AIRCRAFT", "DIAMOND AIRCRAFT INDUSTRY INC"],
    "AVIAT": ["AVIAT AIRCRAFT INC", "AVIAT INC", "AVIAT AIRCRAFT"],
    "AMERICAN CHAMPION": ["AMERICAN CHAMPION AIRCRAFT", "AMERICAN CHAMPION (ACAC)", "AMERICAN CHAMPION AIRCRAFT COR"],
    "PILATUS": ["PILATUS AIRCRAFT LTD", "PILATUS AIRCRAFT"],
    "SOCATA": ["EADS SOCATA"],
    "SWEARINGEN": ["FAIRCHILD SWEARINGEN"],
    "ERCOUPE": ["ERCOUPE (ENG & RESEARCH CORP)"],
    "BRITTEN-NORMAN": ["BRITTEN NORMAN"],
    "CANADAIR": ["CANADAIR LTD"],
}
make_map = {}
for standard_name in variants:
    for variant in variants[standard_name]:
        make_map[variant] = standard_name

df["Make"] = df["Make"].replace(make_map)

# Keep makes with at least 50 so averages are meaningful
make_counts = df["Make"].value_counts()
print("Makes before the 50-accident threshold:", len(make_counts))

def make_count(make):
    return make_counts[make]

df = df[df["Make"].apply(make_count) >= 50]

# The Make column changed, so rebuild the Make.Model identifier
df["Make.Model"] = df["Make"] + " " + df["Model"]
print("Makes kept:", len(df["Make"].unique()))
df["Make"].value_counts().head(15)

Makes before the 50-accident threshold: 923
Makes kept: 47


Make
CESSNA          24939
PIPER           13697
BEECH            4722
MOONEY           1261
GRUMMAN          1218
BOEING            997
BELLANCA          926
AIR TRACTOR       854
AERONCA           558
MAULE             532
CHAMPION          476
CIRRUS            445
STINSON           406
DE HAVILLAND      380
SCHWEIZER         371
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [10]:
df = df.dropna(subset=["Model"])
print("Unique Models:", len(df["Model"].unique()))
print("Unique Make.Models:", len(df["Make.Model"].unique()))
df["Make.Model"].value_counts().head(10)

Unique Models: 1518
Unique Make.Models: 1619


Make.Model
CESSNA 152       2211
CESSNA 172       1614
CESSNA 172N      1088
PIPER PA28140     925
CESSNA 172M       786
CESSNA 150        742
CESSNA 172P       668
PIPER PA18        663
PIPER PA18150     618
CESSNA 182        608
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [11]:
# Engine.Type
print(df["Engine.Type"].value_counts())
df["Engine.Type"] = df["Engine.Type"].replace({"Unknown": np.nan, "UNK": np.nan, "Geared Turbofan": "Turbo Fan"})

# Weather.Condition
print(df["Weather.Condition"].value_counts())
df["Weather.Condition"] = df["Weather.Condition"].replace({"Unk": np.nan, "UNK": np.nan, "Unknown": np.nan})

# Number.of.Engines: 0 engines is impossible for a powered airplane, so it stands for missing
print(df["Number.of.Engines"].value_counts())
df["Number.of.Engines"] = df["Number.of.Engines"].replace(0, np.nan)

# Purpose.of.flight
print(df["Purpose.of.flight"].value_counts())
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace({
    "Unknown": np.nan,
    "Public Aircraft - Federal": "Public Aircraft", "Public Aircraft - State": "Public Aircraft",
    "Public Aircraft - Local": "Public Aircraft", "PUBS": "Public Aircraft",
    "Air Race show": "Air Race/show", "ASHO": "Air Race/show",
})

# Broad.phase.of.flight
print(df["Broad.phase.of.flight"].value_counts())
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].replace("Unknown", np.nan)

# Group very rare categories (fewer than 50 accidents) into "Other"
def group_rare(column, label="Other"):
    counts = df[column].value_counts()
    rare = []
    for category in counts.index:
        if counts[category] < 50:
            rare.append(category)
    df[column] = df[column].replace(rare, label)

group_rare("Engine.Type")
group_rare("Purpose.of.flight")

Engine.Type
Reciprocating      49745
Turbo Prop          2276
Turbo Fan            925
Unknown              397
Turbo Jet            197
Turbo Shaft           13
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64
Weather.Condition
VMC    49684
IMC     4417
UNK      491
Unk      140
Name: count, dtype: int64
Number.of.Engines
1.0    46931
2.0     7086
4.0      109
3.0      107
0.0       85
Name: count, dtype: int64
Purpose.of.flight
Personal                     33567
Instructional                 7864
Aerial Application            3229
Unknown                       3120
Business                      2738
Positioning                    928
Other Work Use                 493
Ferry                          488
Aerial Observation             371
Executive/corporate            301
Public Aircraft                252
Skydiving                      166
Flight Test                    116
Banner Tow                      97
Public Aircraft - Federal       40
Glider Tow    

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [12]:
non_null_counts = df.notna().sum()
print(non_null_counts)

columns_to_drop = []
for column in df.columns:
    if non_null_counts[column] <= 20000:
        columns_to_drop.append(column)

print("Dropping:", columns_to_drop)
df = df.drop(columns=columns_to_drop)
print(df.shape)

Event.Id                  56482
Investigation.Type        56482
Accident.Number           56482
Event.Date                56482
Location                  56461
Country                   56405
Latitude                  21758
Longitude                 21752
Airport.Code              34541
Airport.Name              36124
Injury.Severity           56482
Aircraft.Damage           56482
Aircraft.Category         56482
Registration.Number       55937
Make                      56482
Model                     56482
Amateur.Built             56482
Number.of.Engines         54233
Engine.Type               53157
FAR.Description           17373
Schedule                   6286
Purpose.of.flight         50771
Air.carrier                9592
Total.Fatal.Injuries      56482
Total.Serious.Injuries    56482
Total.Minor.Injuries      56482
Total.Uninjured           56482
Weather.Condition         54101
Broad.phase.of.flight     40284
Report.Status             53396
Publication.Date          46998
Make.Mod

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [13]:
df.to_csv("data/AviationData_cleaned.csv", index=False)